# 2d. Dynamic Parameters, Hard-A After Uniform

**Scenario**: Animals reach expert on Uniform, then switch to Hard-A.
After the switch, parameters spike (re-learning) then settle back.
This is the most realistic synthetic scenario.

**Key question**: Can model ID work during and after adaptation
to a new distribution? This validates 4b's per-phase assignment.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_NOTEBOOK_ROOT = _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _NOTEBOOK_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
if str(_NOTEBOOK_ROOT) not in sys.path:
	sys.path.insert(0, str(_NOTEBOOK_ROOT))

from shared_setup import *

import time
import pickle
from analysis.validation import (
    make_synthetic_cohort, make_learning_cohort, make_shift_cohort,
    run_gs_model_id, run_sbi_model_id, summarise_agreement,
)

SBI_OK = False
try:
    import torch
    SBI_OK = True
    print(f'SBI available (torch={torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

## 1. Configuration

In [ ]:
QUICK = True

if QUICK:
    N_SYNTHETIC = 3; N_GS_SEEDS = 2; BURN_IN = 500
    N_SBI_SIMS = 1_000; N_GENERIC_TRIALS = 300; N_CV_REPEATS = 4
else:
    N_SYNTHETIC = 5; N_GS_SEEDS = 4; BURN_IN = 1000
    N_SBI_SIMS = 20_000; N_GENERIC_TRIALS = 2500; N_CV_REPEATS = 32

SEED = 42
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

N_UNIFORM = 10
N_HARD_A = 10
TRIALS_PER_SESSION = 350

## 2. Generate Dynamic-Shift Animals

In [ ]:
animals = make_shift_cohort(
    n_per_model=N_SYNTHETIC, n_uniform=N_UNIFORM, n_hard_a=N_HARD_A,
    trials_per_session=TRIALS_PER_SESSION,
    burn_in=BURN_IN, dynamic_hard_a=True, seed=SEED,
)
print(f'{len(animals)} animals')
for a in animals:
    u_accs = [s.stats(['accuracy'])['accuracy'] for s in a['uniform_sessions'][-3:]]
    ha_accs = [s.stats(['accuracy'])['accuracy'] for s in a['hard_a_sessions']]
    print(f'  {a["animal_id"]}: {a["true_model"]} '
          f'expert-uniform={np.mean(u_accs):.2f} '
          f'hard-a=[{ha_accs[0]:.2f}→{ha_accs[-1]:.2f}]')

## 3. Model ID on Hard-A Sessions (During Adaptation)

The Hard-A sessions include the initial re-learning period.
Parameters are changing — this is the hardest test.

In [ ]:
print('=== GS on Hard-A sessions ===')
gs_df = run_gs_model_id(
    animals, sessions_key='hard_a_sessions',
    n_seeds=N_GS_SEEDS, burn_in=BURN_IN,
)

In [ ]:
sbi_df = pd.DataFrame()
if SBI_OK:
    print('=== SBI on Hard-A sessions ===')
    sbi_df = run_sbi_model_id(
        animals, sessions_key='hard_a_sessions',
        n_sbi_sims=N_SBI_SIMS, n_generic_trials=N_GENERIC_TRIALS,
        n_cv_repeats=N_CV_REPEATS, burn_in=BURN_IN, seed=SEED,
    )

## 4. Agreement

In [ ]:
merged = summarise_agreement(gs_df, sbi_df, 'Dynamic Hard-A (re-learning)')
print()
print(merged.to_string(index=False))

## 5. Save

In [ ]:
# Save results for 2e agreement summary
save_path = Path('../results/validation/2d_dynamic_hard_a.pkl')
save_path.parent.mkdir(parents=True, exist_ok=True)
with open(save_path, 'wb') as f:
    pickle.dump({'gs': gs_df, 'sbi': sbi_df if SBI_OK else None,
                'scenario': '2d_dynamic_hard_a'
}, f)
print(f'Saved to {save_path}')